In [ ]:
#@title ⚙️ Setup — choose how to provide the inputs (RUN ME FIRST)
#@markdown Pick how the input files for this step are provided:
#@markdown - **Mobile mode**: the files are downloaded automatically from the GitHub repo — no manual upload, works on phones/tablets.
#@markdown - **Manual upload**: you upload the files yourself in the cells below.
input_mode = "Mobile mode (auto-download from repo)" #@param ["Mobile mode (auto-download from repo)", "Manual upload"]

import os, urllib.request
REPO_RAW = "https://raw.githubusercontent.com/biochorl/Nanobody_de_novo_design/main"

# Inputs needed by THIS step.  (some links are FACSIMILE placeholders until the files
# are added to the repo — they will start working once they are committed)
INPUT_FILES = [('/content/target_antigen.pdb', '/Example_input/7z1b.pdb', 'real'), ('/content/nanobody_scaffold.pdb', '/Example_input/nanobody_scaffold.pdb', 'real'), ('/content/discotope_output.pdb', '/Example_output/antigen_A.pdb', 'facsimile'), ('/content/discotope_output.csv', '/Example_output/file_A_discotope3.csv', 'facsimile')]

def fetch_inputs(files):
    ok = 0
    for dst, rel, kind in files:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        url = REPO_RAW + rel
        try:
            urllib.request.urlretrieve(url, dst)
            print(f"✅ {os.path.basename(dst):<28} ← {url}  [{kind}]")
            ok += 1
        except Exception as e:
            tag = "facsimile not in repo yet" if kind == "facsimile" else "ERROR"
            print(f"⚠️  {os.path.basename(dst):<28} could not be downloaded ({tag}): {e}")
    return ok

if input_mode.startswith("Mobile"):
    print("📱 Mobile mode — downloading this step's inputs from the repo:\n")
    n = fetch_inputs(INPUT_FILES)
    print(f"\n→ {n}/{len(INPUT_FILES)} file(s) ready in /content. "
          "Skip the manual-upload cells and point the path fields to these files.")
else:
    print("🖱️ Manual mode — upload your files in the cells below as usual.")


# **IMPORTANT**: Set the runtime to CPU-only

This notebook is designed to run on a CPU-only runtime. Using a GPU will not provide any performance benefits.

**How to set the runtime to CPU-only:**
1. Go to `Runtime` in the top menu.
2. Select `Change runtime type`.
3. In the `Hardware accelerator` dropdown, select `CPU`.
4. Click `Save`.

#1) **DiscoTope-3.0**: identification of putative epitopes on the surface of the target antigen, using the [DiscoTope-3.0 web server](https://services.healthtech.dtu.dk/services/DiscoTope-3.0/).

## Define epitopes with DiscoTope-3.0

DiscoTope-3.0 runs on the DTU Health Tech web server, so there is nothing heavy to install locally — you just submit your antigen structure and load the results here.

**Instructions**
1. Open DiscoTope-3.0: https://services.healthtech.dtu.dk/services/DiscoTope-3.0/
2. Upload your **target antigen** PDB and follow the site instructions.
3. Download the results and unzip them: you get a predicted **.pdb** and a per-residue **.csv**.
4. Run the cell below and, when prompted, upload **the DiscoTope output `.pdb`** (recommended — its chain and numbering match the CSV) and the DiscoTope **`.csv`**.
   - If you only have the original PDB, the cell will still work: it aligns the antigen **by sequence**, so a different chain ID or residue numbering is handled automatically.
5. The cell extracts the antigen (saved as single chain `A` → `antigen_A.pdb` / `antigen.fasta`), clusters the predicted epitope residues into 3D "hotspots", and prints, for each cluster, the residues **as 0-based positional indices** — which is exactly what IgGM's `--epitope` expects (it indexes into the antigen sequence, not PDB residue numbers).

In [ ]:
# @title Calculate, cluster and visualize DiscoTope-3.0 epitopes (upload DiscoTope .pdb + .csv)
!pip install -q py3Dmol
import os, csv, collections
import numpy as np
import py3Dmol
from google.colab import files

THREE_TO_ONE = {'ALA':'A','CYS':'C','ASP':'D','GLU':'E','PHE':'F','GLY':'G','HIS':'H','ILE':'I','LYS':'K',
                'LEU':'L','MET':'M','ASN':'N','PRO':'P','GLN':'Q','ARG':'R','SER':'S','THR':'T','VAL':'V',
                'TRP':'W','TYR':'Y'}

#@markdown **Cutoff Distance (Å):** residues within this distance are grouped into the same epitope cluster.
cluster_cutoff = 6.0 #@param {type:"number"}

def aa1(x):
    x = str(x).strip()
    return x.upper() if len(x) == 1 else THREE_TO_ONE.get(x.upper(), 'X')

def col_index(header, *names):
    low = [h.strip().lower() for h in header]
    for n in names:
        if n in low:
            return low.index(n)
    return None

def parse_discotope_csv(path):
    """Return OrderedDict: chain -> ordered list of dicts {res_id, aa, epi} (one row per residue)."""
    by_chain = collections.OrderedDict()
    with open(path, newline='') as fh:
        reader = csv.reader(fh)
        header = next(reader)
        ci = col_index(header, 'chain')
        ri = col_index(header, 'res_id', 'residue_id', 'resi', 'pos', 'position')
        ni = col_index(header, 'residue', 'restype', 'aa', 'amino_acid', 'res_name')
        ei = col_index(header, 'epitope', 'is_epitope')
        if None in (ci, ri, ei):
            print(f"❌ CSV missing required columns. Found header: {header}")
            return by_chain
        for row in reader:
            if len(row) <= max(ci, ri, ni or 0, ei):
                continue
            ch = row[ci].strip()
            try:
                rid = int(float(row[ri]))
            except ValueError:
                continue
            aa = aa1(row[ni]) if ni is not None else 'X'
            epi = str(row[ei]).strip().lower() in ('true', '1', 'yes')
            by_chain.setdefault(ch, []).append({'res_id': rid, 'aa': aa, 'epi': epi})
    return by_chain

def parse_pdb_chains(pdb_text):
    """Return OrderedDict: chain -> ordered list of (res_id, aa, ca_xyz)."""
    chains = collections.OrderedDict()
    seen = set()
    for line in pdb_text.splitlines():
        if line.startswith('ATOM') and len(line) > 54 and line[12:16].strip() == 'CA':
            ch = line[21]
            resn = line[17:20].strip()
            resi = int(line[22:26])
            key = (ch, resi)
            if key in seen or resn not in THREE_TO_ONE:
                continue
            seen.add(key)
            xyz = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            chains.setdefault(ch, []).append((resi, THREE_TO_ONE[resn], xyz))
    return chains

def cluster_residues(coords, cutoff):
    items = list(coords)  # list of (key, xyz)
    n = len(items); un = list(range(n)); clusters = []
    while un:
        cur = [un.pop(0)]; q = [cur[0]]
        while q:
            i = q.pop(0)
            near = [j for j in un if np.linalg.norm(items[i][1] - items[j][1]) <= cutoff]
            for j in near:
                un.remove(j); cur.append(j); q.append(j)
        clusters.append([items[i][0] for i in cur])
    return clusters

print('Please upload the DiscoTope output PDB (or your antigen PDB).')
up_pdb = files.upload()
print('\nPlease upload the DiscoTope-3.0 output CSV.')
up_csv = files.upload()

if up_pdb and up_csv:
    antigen_pdb_name = next(iter(up_pdb))
    antigen_pdb_content = up_pdb[antigen_pdb_name].decode('utf-8', errors='ignore')
    discotope_csv_name = next(iter(up_csv))
    with open(discotope_csv_name, 'wb') as f:
        f.write(up_csv[discotope_csv_name])

    csv_by_chain = parse_discotope_csv(discotope_csv_name)
    pdb_chains = parse_pdb_chains(antigen_pdb_content)
    if not csv_by_chain or not pdb_chains:
        raise SystemExit('❌ Could not parse the CSV and/or PDB. Check the uploaded files.')

    # CSV chain to use = the one with epitope residues (fallback: first)
    csv_chain = max(csv_by_chain, key=lambda ch: sum(r['epi'] for r in csv_by_chain[ch]))
    csv_rows = csv_by_chain[csv_chain]
    csv_seq = ''.join(r['aa'] for r in csv_rows)
    n_epi = sum(r['epi'] for r in csv_rows)
    print(f"ℹ️ DiscoTope CSV: chain '{csv_chain}', {len(csv_rows)} residues, {n_epi} predicted epitope residues.")

    # match the PDB chain BY SEQUENCE (robust to different chain IDs / numbering)
    pdb_chain = None
    for ch, residues in pdb_chains.items():
        if ''.join(a for _, a, _ in residues) == csv_seq:
            pdb_chain = ch
            break
    if pdb_chain is None:
        if len(pdb_chains) == 1:
            pdb_chain = next(iter(pdb_chains))
            print(f"⚠️ No exact sequence match; using the single PDB chain '{pdb_chain}'. "
                  f"(PDB {len(pdb_chains[pdb_chain])} res vs CSV {len(csv_rows)} res — best-effort positional alignment.)")
        else:
            raise SystemExit("❌ Could not match the CSV antigen to a PDB chain by sequence. "
                             "Please upload the DiscoTope OUTPUT PDB (same chain & numbering as the CSV).")
    else:
        print(f"✅ Matched DiscoTope chain '{csv_chain}' to PDB chain '{pdb_chain}' by sequence.")

    pdb_residues = pdb_chains[pdb_chain]   # ordered (res_id, aa, ca)
    antigen_seq = ''.join(a for _, a, _ in pdb_residues)

    # write single-chain antigen as chain 'A' (coords preserved, no SEQRES -> IgGM reads ATOM order)
    out_lines = []
    for line in antigen_pdb_content.splitlines():
        if line.startswith(('ATOM', 'HETATM', 'TER', 'ANISOU')) and len(line) > 21 and line[21] == pdb_chain:
            out_lines.append(line[:21] + 'A' + line[22:])
    out_lines.append('END')
    with open('antigen_A.pdb', 'w') as f:
        f.write('\n'.join(out_lines) + '\n')
    with open('antigen.fasta', 'w') as f:
        f.write('>A\n' + antigen_seq + '\n')
    print(f"\U0001f9ec Antigen -> single chain 'A' (antigen_A.pdb), length {len(antigen_seq)}.")

    # epitope entries: positional index (0-based, what IgGM --epitope expects) + PDB res_id + CA coord
    epi_entries = []
    for pos, row in enumerate(csv_rows):
        if row['epi'] and pos < len(pdb_residues):
            rid, _, ca = pdb_residues[pos]
            epi_entries.append({'pos': pos, 'res_id': rid, 'ca': ca})

    clusters = cluster_residues([((e['pos'], e['res_id']), e['ca']) for e in epi_entries], cluster_cutoff)
    colors = ['red','blue','green','orange','purple','yellow','cyan','magenta','salmon','lime','teal','gold','violet','pink']
    print(f"✅ Clustered {len(epi_entries)} epitope residues into {len(clusters)} hotspot(s).")

    print("\n--- Epitope hotspots: 0-based positional indices for IgGM --epitope ---")
    with open('Epitopes.txt', 'w') as f:
        f.write("# values below are 0-based positional indices into the antigen sequence (IgGM --epitope)\n")
        for i, cl in enumerate(clusters):
            positions = sorted(p for (p, rid) in cl)
            pdb_nums = sorted(rid for (p, rid) in cl)
            line = f"epitope {i+1} ({colors[i % len(colors)]}): {' '.join(map(str, positions))}"
            print(line + f"   [PDB res ids: {' '.join(map(str, pdb_nums))}]")
            f.write(line + f"   # PDB res ids: {' '.join(map(str, pdb_nums))}\n")
    print("----------------------------------------------------------------------")

    print("\nGenerating 3D visualization (epitope hotspots highlighted)...")
    view = py3Dmol.view(width=800, height=600)
    view.addModel(antigen_pdb_content, 'pdb')
    view.setStyle({'cartoon': {'color': 'lightgray'}})
    for i, cl in enumerate(clusters):
        col = colors[i % len(colors)]
        sel = {'or': [{'chain': pdb_chain, 'resi': rid} for (p, rid) in cl]}
        view.addStyle(sel, {'stick': {'color': col, 'radius': 0.2}, 'sphere': {'scale': 0.4, 'color': col}})
    view.zoomTo()
    view.show()
else:
    print('❌ You must upload BOTH the DiscoTope PDB and the DiscoTope CSV.')


--------------------
# 2) **Nanocdr-x**: identification of CDRs on the PDB structure of the nanobody scaffold. Based on the content of [this](https://github.com/lescailab/nanocdr-x) GitHub repo

##Map Nanobody CDRs (when asked to upload, please uplod the nanobody scaffold. Please re-run the colab notebook from the beginning if it disconnects during execution)

In [ ]:
# @title Map Nanobody CDRs  ->  IgGM-aligned output: >H nanobody FASTA with CDRs = X
#@markdown ### CDR masking
#@markdown The nanobody CDRs are masked at their **native length** (the detected length is kept here).
#@markdown To explore **different CDR3 loop lengths**, use the **CDR3 length interval** in Notebook 2
#@markdown (suggested nanobody CDR3 range ≈ **7–20** residues) — IgGM samples one length per design.
!pip install -q condacolab
import condacolab
condacolab.install()

!conda install -c lescailab nanocdr-x -y

import pandas as pd
from google.colab import files
import os

three_to_one = {'ALA': 'A', 'CYS': 'C', 'ASP': 'D', 'GLU': 'E', 'PHE': 'F',
                'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LYS': 'K', 'LEU': 'L',
                'MET': 'M', 'ASN': 'N', 'PRO': 'P', 'GLN': 'Q', 'ARG': 'R',
                'SER': 'S', 'THR': 'T', 'VAL': 'V', 'TRP': 'W', 'TYR': 'Y'}

def get_sequence_from_pdb(pdb_string):
    sequence, residue_info, seen = [], [], set()
    for line in pdb_string.splitlines():
        if line.startswith('ATOM') and line[13:15].strip() == 'CA':  # alpha carbons
            res_id = int(line[22:26]); res_name = line[17:20]
            if res_id not in seen:
                seen.add(res_id)
                if res_name in three_to_one:
                    sequence.append(three_to_one[res_name])
                    residue_info.append((res_id, three_to_one[res_name]))
    return "".join(sequence), residue_info

print('Please upload the nanobody scaffold PDB (the framework you want to redesign).')
uploaded_nb = files.upload()

if not uploaded_nb:
    print('No nanobody file uploaded.')
else:
    nanobody_pdb_name = list(uploaded_nb.keys())[0]
    nanobody_pdb_content = uploaded_nb[nanobody_pdb_name].decode('utf-8')
    full_sequence, residue_map = get_sequence_from_pdb(nanobody_pdb_content)

    # Run nanocdr-x (CDR detection unchanged)
    input_df = pd.DataFrame([{'identifier': nanobody_pdb_name, 'input': full_sequence}])
    input_csv_path = 'nanobody_input.csv'
    output_csv_path = 'nanobody_cdrs.csv'
    input_df.to_csv(input_csv_path, index=False)
    !predict_cdrs -i {input_csv_path} -o {output_csv_path}

    if os.path.exists(output_csv_path):
        results_df = pd.read_csv(output_csv_path)
        all_cdr_res_ids = {}
        for i in range(1, 4):
            cdr_seq = results_df[f'predicted_cdr{i}'][0]
            if pd.notna(cdr_seq):
                start_index = full_sequence.find(cdr_seq)
                if start_index != -1:
                    cdr_residues, cdr_res_ids = [], []
                    for j in range(len(cdr_seq)):
                        res_id, res_name = residue_map[start_index + j]
                        cdr_residues.append(f'{res_name}{res_id}')
                        cdr_res_ids.append(res_id)
                    all_cdr_res_ids[i] = cdr_res_ids
                    print(f'CDR{i}: {",".join(cdr_residues)}')
                else:
                    print(f'CDR{i}: Could not map sequence {cdr_seq} to the original structure.')
            else:
                print(f'CDR{i}: Not found.')

        # --- IgGM requirement: nanobody as a single chain 'H' with the CDRs (redesign regions) masked as 'X' ---
        # Per-CDR redesign length (0 = keep native length); changes the number of X so IgGM designs loops of that length.
        cdr_len_target = {1: 0, 2: 0, 3: 0}  # native CDR lengths; CDR3 length range is explored in Notebook 2
        resid_to_cdr = {}
        for _k, _ids in all_cdr_res_ids.items():
            for _r in _ids:
                resid_to_cdr[_r] = _k
        masked_chars = []
        _i = 0
        while _i < len(residue_map):
            res_id, aa = residue_map[_i]
            cdr = resid_to_cdr.get(res_id)
            if cdr is None:
                masked_chars.append(aa)
                _i += 1
            else:
                # consume the whole contiguous CDR block, then emit the chosen number of X
                block = 0
                while _i < len(residue_map) and resid_to_cdr.get(residue_map[_i][0]) == cdr:
                    block += 1
                    _i += 1
                n_x = cdr_len_target.get(cdr) or block  # 0/None -> native length
                masked_chars.append('X' * int(n_x))
        masked_nb_seq = ''.join(masked_chars)
        for _c in (1, 2, 3):
            _tgt = cdr_len_target.get(_c) or 0
            if _tgt:
                print(f"   CDR{_c}: redesign length set to {_tgt} (native was {len(all_cdr_res_ids.get(_c, []))})")
        nb_chain_id = 'H'  # IgGM convention for the (nano)antibody chain
        with open('nanobody_redesign.fasta', 'w') as f:
            f.write(f'>{nb_chain_id}\n{masked_nb_seq}\n')
        print(f"\n✅ IgGM nanobody FASTA saved: nanobody_redesign.fasta  (chain {nb_chain_id}, X={masked_nb_seq.count('X')})")
        print(f">{nb_chain_id}\n{masked_nb_seq}")
        print("\nTip: you can edit the CDR lengths by adding/removing 'X', or keep a residue fixed by "
              "restoring its letter in place of the 'X'.")
    else:
        print('CDR prediction output file not found.')


--------------------
# 3) **Combined IgGM input FASTA**

A fasta of the nanobody (">H" header) with CDRs masked by "X" and the target antigen (">A" header) are merged into `IgGM_input.fasta` with the **nanobody first and the antigen last**, exactly as IgGM expects

In [ ]:
# @title Generate combined IgGM input FASTA
import os
from google.colab import files

def read_fasta(path):
    recs, h, buf = [], None, []
    for line in open(path):
        line = line.strip()
        if not line:
            continue
        if line.startswith('>'):
            if h is not None:
                recs.append((h, ''.join(buf)))
            h, buf = line[1:].strip(), []
        else:
            buf.append(line)
    if h is not None:
        recs.append((h, ''.join(buf)))
    return recs

nb_ok = os.path.exists('nanobody_redesign.fasta')
ag_ok = os.path.exists('antigen.fasta')
if not nb_ok:
    print("⚠️ nanobody_redesign.fasta not found — run the 'Map Nanobody CDRs' cell first.")
if not ag_ok:
    print("⚠️ antigen.fasta not found — run the DiscoTope epitope cell first.")

if nb_ok and ag_ok:
    nb_id, nb_seq = read_fasta('nanobody_redesign.fasta')[0]
    ag_id, ag_seq = read_fasta('antigen.fasta')[0]
    nb_id = nb_id or 'H'
    ag_id = ag_id or 'A'
    if nb_id.upper() == ag_id.upper():
        nb_id = 'H' if ag_id.upper() != 'H' else 'B'
    with open('IgGM_input.fasta', 'w') as f:
        f.write(f'>{nb_id}\n{nb_seq}\n')   # nanobody first (X = redesign)
        f.write(f'>{ag_id}\n{ag_seq}\n')   # antigen LAST
    print("✅ Combined IgGM input FASTA saved: IgGM_input.fasta")
    print(f">{nb_id}   (nanobody, redesign = X, len {len(nb_seq)}, X={nb_seq.count('X')})")
    print(f">{ag_id}   (antigen, len {len(ag_seq)})")
    files.download('IgGM_input.fasta')
    if os.path.exists('antigen_A.pdb'):
        files.download('antigen_A.pdb')
    # epitope annotations from the DiscoTope step
    for _epi in ('Epitopes.txt', 'antigen.fasta'):
        if os.path.exists(_epi):
            files.download(_epi)
    print("\n➡️ For the IgGM design step use:  --fasta IgGM_input.fasta   --antigen antigen_A.pdb")


---
### ✅ Step complete — disconnect before the next notebook

To free the GPU/CPU for the next step, **disconnect and delete this runtime**:
`Runtime → Disconnect and delete runtime` (or `Manage sessions → Terminate`).

⬅️ **[Back to the main workflow](https://biochorl.github.io/Nanobody_de_novo_design/)**
